In [ ]:
!pip install pandas dask

# import

In [ ]:
sample = pd.read_csv('./datasets/sample_submission.csv')
train = pd.read_csv('./datasets/train.csv')
test = pd.read_csv('./datasets/test.csv')

In [ ]:
# 가로 너비 제한으로 인해 데이터프레임이 다음 줄로 넘어가는 것(줄바꿈)을 방지
pd.set_option('display.expand_frame_repr', False)

# 모든 컬럼을 생략 없이 출력하도록 설정
pd.set_option('display.max_columns', None)

# 텍스트 출력 콘솔의 가로 최대 너비를 충분히 크게 설정
pd.set_option('display.width', 1000)

# 데이터 정합성 검사

### 행 수 조사

In [ ]:
print("sample: ", len(sample))
print("train :", len(train))
print("test :", len(test))

<b>->train set가 많은 것에 비해 test set이 너무 적음, train set의 일부를 test로 재가공하거나 K-fold 교차 검증해야 할 필요</b>

### 결측치 확인

In [ ]:
# orders.csv 결측치
print(f"sample 결측치:\n{sample.isnull().sum()}\n")

# order_products__prior.csv 결측치
print(f"train 결측치:\n{train.isnull().sum()}\n")

# order_products__train.csv 결측치
print(f"test 결측치:\n{test.isnull().sum()}\n")

### 샘플링

In [ ]:
sample.head()

<b>-> 모델 출력에 softmax 써서 모든 항목 확률 합이 1이 되도록 구성해야 함.</b>

In [ ]:
train.head()

In [ ]:
test.head()

<b>-> test 행은 정답 컬럼 X, 모델 I/O 포매팅 용으로 준듯</b>

### model 행 unique 나열

In [ ]:
# 1. model_a 컬럼의 고유값 확인
unique_model_a = train['model_a'].unique()
print("=== model_a 고유값 ===")
print(f"개수: {len(unique_model_a)}개")
print(unique_model_a)
print("\n")

In [ ]:
# 2. model_b 컬럼의 고유값 확인
unique_model_b = train['model_b'].unique()
print("=== model_b 고유값 ===")
print(f"개수: {len(unique_model_b)}개")
print(unique_model_b)
print("\n")

In [ ]:
# 두 컬럼의 데이터를 하나의 시리즈로 합친 후 고유값 추출
all_models = pd.concat([train['model_a'], train['model_b']])
unique_all_models = all_models.unique()

print("=== model_a, model_b 전체 고유값 (중복 제거) ===")
print(f"총 고유 모델 개수: {len(unique_all_models)}개")
print(unique_all_models)

### winner 행 모순 조사 - (1, 0, 0), (0, 1, 0) (0, 0, 1)외의 행이 존재하는지

In [ ]:
# 검증할 타겟 컬럼 리스트
target_cols = ['winner_model_a', 'winner_model_b', 'winner_tie']

# 1. train 데이터프레임 검증
# 각 행(axis=1)별로 세 컬럼의 값을 더했을 때 1이 아닌 행을 필터링
invalid_rows_train = train[train[target_cols].sum(axis=1) != 1]

print("=== train 데이터셋 검증 결과 ===")
if invalid_rows_train.empty:
    print("결과: 정상입니다. 모든 행이 유효한 패턴을 가지고 있습니다.\n")
else:
    print(f"결과: 예외 데이터가 {len(invalid_rows_train)}건 발견되었습니다.")
    print("예외 데이터 상위 5건 출력:")
    print(invalid_rows_train[target_cols].head())
    print("\n")

# 데이터 탐색

### 모델 응답값 길이 조사 - LLM 사용 시 max_seq_len에 반영

In [ ]:
def print_min_max_text(df, col):
    # 해당 컬럼의 결측치가 없는 데이터만 필터링
    valid_df = df.dropna(subset=[col])
    
    # 각 문자열의 길이 계산
    lengths = valid_df[col].str.len()
    
    # 최대값과 최소값을 가지는 행의 인덱스 추출
    max_idx = lengths.idxmax()
    min_idx = lengths.idxmin()
    
    # 실제 텍스트 데이터 추출 (글자 수 제한 없음)
    max_text = valid_df.loc[max_idx, col]
    min_text = valid_df.loc[min_idx, col]
    
    print(f"=== [{col}] 최소 길이 예시 (길이: {lengths[min_idx]}) ===")
    print(min_text)
    print("\n" + "="*80 + "\n")
    
    print(f"=== [{col}] 최대 길이 예시 (길이: {lengths[max_idx]}) ===")
    print(max_text)
    print("\n")

In [ ]:
print_min_max_text(train, 'prompt')

In [ ]:
print_min_max_text(train, 'response_a')

In [ ]:
print_min_max_text(train, 'response_b')

<b>-> 최소 길이 응답 거부인거 같은데 이 경우 사용자가 해당 모델을 고를 리가 없음, 이 부분은 하드코딩해야 할 듯</b>

-> 마찬가지로 최대 길이도 저런 응답을 고를 리가 없음

?

### 이상한 모델 조사 - 항상 지는 모델이 있는지, 승률이 가장 낮은 모델은 확률이 얼마인지

In [ ]:
# [Cell 7] 64개 모델 전체 승률 계산 및 정렬 (상위 10개, 하위 10개 출력)
# 1. model_a 기준 통계
stats_a = train.groupby('model_a').agg(
    matches_as_a=('model_a', 'count'),
    wins_as_a=('winner_model_a', 'sum'),
    ties_as_a=('winner_tie', 'sum')
).reset_index().rename(columns={'model_a': 'model'})

# 2. model_b 기준 통계
stats_b = train.groupby('model_b').agg(
    matches_as_b=('model_b', 'count'),
    wins_as_b=('winner_model_b', 'sum'),
    ties_as_b=('winner_tie', 'sum')
).reset_index().rename(columns={'model_b': 'model'})

# 3. 병합 및 전체 총합 산출
model_stats = pd.merge(stats_a, stats_b, on='model', how='outer').fillna(0)
model_stats['total_matches'] = model_stats['matches_as_a'] + model_stats['matches_as_b']
model_stats['total_wins'] = model_stats['wins_as_a'] + model_stats['wins_as_b']
model_stats['total_ties'] = model_stats['ties_as_a'] + model_stats['ties_as_b']

# 4. 승률 계산
model_stats['win_rate(%)'] = (model_stats['total_wins'] / model_stats['total_matches']) * 100

# 5. 정렬 및 최종 정리
final_stats = model_stats[['model', 'total_matches', 'total_wins', 'total_ties', 'win_rate(%)']]
final_stats = final_stats.sort_values(by='win_rate(%)', ascending=False).reset_index(drop=True)

# 6. 결과 출력
print(f"집계된 총 모델 개수: {final_stats.shape[0]}개\n")

print("=== 모델별 승률 통계 (상위 10개) ===")
print(final_stats.head(10))
print("\n")

print("=== 모델별 승률 통계 (하위 10개) ===")
print(final_stats.tail(10))

<b>모델 자체가 승률이 심각하게 낮은 경우, 높은 경우도 있음</b>

### 자리에 의한 편향 조사 - 각 모델이 1번 자리에 위치했을 때와 2번 자리에 위치했을 때의 승률 차이  

In [ ]:
# 1. 전체 데이터셋 기준 자리 편향 확인
total_matches = len(train)
total_win_a = train['winner_model_a'].sum()
total_win_b = train['winner_model_b'].sum()
total_tie = train['winner_tie'].sum()

print("=== 1. 전체 데이터셋 기준 자리 편향 ===")
print(f"전체 경기 수: {total_matches}")
print(f"- 1번 자리(model_a) 승리: {total_win_a}회 ({total_win_a/total_matches*100:.2f}%)")
print(f"- 2번 자리(model_b) 승리: {total_win_b}회 ({total_win_b/total_matches*100:.2f}%)")
print(f"- 무승부: {total_tie}회 ({total_tie/total_matches*100:.2f}%)\n")

In [ ]:
# 2. 모델별 자리 편향 계산
# model_a로 출전했을 때의 경기 수와 승리 수
stats_a = train.groupby('model_a').agg(
    matches_as_a=('model_a', 'count'),
    wins_as_a=('winner_model_a', 'sum')
).reset_index().rename(columns={'model_a': 'model'})

# model_b로 출전했을 때의 경기 수와 승리 수
stats_b = train.groupby('model_b').agg(
    matches_as_b=('model_b', 'count'),
    wins_as_b=('winner_model_b', 'sum')
).reset_index().rename(columns={'model_b': 'model'})

# 두 통계를 병합
bias_df = pd.merge(stats_a, stats_b, on='model', how='outer').fillna(0)

# 각 자리에서의 승률 계산 (%)
bias_df['win_rate_a(%)'] = (bias_df['wins_as_a'] / bias_df['matches_as_a']) * 100
bias_df['win_rate_b(%)'] = (bias_df['wins_as_b'] / bias_df['matches_as_b']) * 100

# 승률 차이 계산: 1번 자리 승률 - 2번 자리 승률
# 양수(+)면 1번 자리일 때 승률이 높음, 음수(-)면 2번 자리일 때 승률이 높음을 의미
bias_df['bias_diff(%p)'] = bias_df['win_rate_a(%)'] - bias_df['win_rate_b(%)']

# 자리 편향이 가장 심한 모델을 찾기 위해 차이의 절대값을 기준으로 내림차순 정렬
bias_df['abs_diff'] = bias_df['bias_diff(%p)'].abs()
bias_df = bias_df.sort_values(by='abs_diff', ascending=False).reset_index(drop=True)

# 출력 시 절대값 컬럼은 불필요하므로 제외하고 필요한 컬럼만 선택
final_bias_df = bias_df[['model', 'matches_as_a', 'win_rate_a(%)', 'matches_as_b', 'win_rate_b(%)', 'bias_diff(%p)']]

print("=== 2. 모델별 자리 편향 통계 (편향이 가장 심한 상위 15개) ===")
print("주의: bias_diff(%p)가 양수면 1번 자리에서 유리, 음수면 2번 자리에서 유리했음을 의미합니다.")
display(final_bias_df.head(15))